<style>
.jp-Notebook { max-width: 1000px; margin: 0 auto; }
.jp-RenderedMarkdown h1 { border-bottom: 2px solid #ddd; padding-bottom: .3em; }
.jp-RenderedMarkdown h2 { margin-top: 1.8em; color: #123; border-bottom: 1px solid #eee; }
.jp-RenderedMarkdown h3 { margin-top: 1.4em; color: #345; }
.jp-RenderedHTMLCommon table { margin: 1em 0; }
</style>

# Atividade 2 — Perceptron: Separabilidade e Limitações

**Redes Neurais — Insper, 2026.2**

O fio condutor da atividade é a **separabilidade linear**. O mesmo perceptron, escrito uma única
vez, é treinado em dois datasets: um que o algoritmo resolve (Exercício 1) e outro que ele não
consegue resolver (Exercício 2). O interessante não é que o segundo falhe, e sim **como** ele
falha.

**Conformidade com as regras técnicas.** Semente fixa via `rng = np.random.default_rng(42)`,
com **um único objeto `rng` reutilizado em todo o relatório**. O notebook é executado de cima
para baixo (`jupyter nbconvert --execute`), então a sequência de sorteios e todos os números
desta página são reprodutíveis. Os sorteios, em ordem, são: dados do Ex. 1 (2 sorteios),
$\mathbf{w}$ inicial do Ex. 1, dados do Ex. 2 (2 sorteios), $\mathbf{w}$ inicial do Ex. 2 e, por
último, a permutação do experimento complementar do item 2.D. Bibliotecas: **NumPy**, **pandas** e
**Matplotlib**. A ativação, a predição, a regra de atualização e o laço de treino foram escritos
à mão. **Nenhuma biblioteca fornece o modelo**: não há scikit-learn neste relatório.

**Uso de ferramentas de IA.** Este relatório foi produzido com apoio do Claude (Anthropic) na
escrita do código, na geração das figuras e na redação das análises. As decisões metodológicas,
os parâmetros e as interpretações foram revisados e validados por mim, e consigo explicar cada
trecho do código e cada conclusão apresentada. Todo número pedido pelo enunciado está escrito no
texto, e não apenas na saída das células.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# ÚNICO gerador aleatório do relatório: criado aqui e reutilizado em todas as
# amostragens abaixo, na ordem em que as células aparecem.
rng = np.random.default_rng(42)

N_PER_CLASS = 1000          # 1000 amostras por classe -> 2000 no total
CORES = ["#1f77b4", "#d62728"]
ROTULOS = ["Classe 0", "Classe 1"]

## Abordagem de implementação e desafios

**Implementação.** O perceptron é uma única função, `treinar_perceptron`, usada sem nenhuma
alteração nos dois exercícios. O laço percorre as amostras uma a uma, porque a regra é
sequencial: cada atualização muda os pesos usados na predição seguinte, então o laço *não* pode
ser vetorizado entre amostras. Já a acurácia no dataset inteiro, recalculada várias vezes, é
vetorizada (`X @ w + b`). Os pesos iniciais entram como argumento (`w_init`) em vez de serem
sorteados dentro da função. Assim, as duas execuções do item 1.D partem exatamente do mesmo
ponto, e o $\eta$ é a única diferença entre elas. O *pocket* (melhor $(\mathbf{w}, b)$ já visto)
está na função desde o início. No Exercício 1 ele não muda nada no treino; o custo é só reavaliar
a acurácia a cada atualização.

**Convenções adotadas.**

- $\text{step}(0) = 1$, como no enunciado ($z \geq 0$).
- **Ordem das amostras:** os dados ficam como foram gerados (as 1000 amostras da Classe 0 e
  depois as 1000 da Classe 1), na mesma ordem fixa em toda época, sem embaralhar.
- **Contagem de épocas:** o número reportado é o de épocas *executadas*, incluindo a última
  passada limpa, aquela que comprova que não há mais atualizações.

**Desafios.**

1. *A ordem das amostras.* O enunciado não a especifica, e no Exercício 2 ela muda radicalmente
   o resultado dos pesos finais. Foi mantida a ordem natural de geração, e o efeito de embaralhar
   é medido num experimento complementar ao final do item 2.D, que roda por último para não
   alterar nenhum sorteio anterior.
2. *Por que $\eta$ quase não importa.* À primeira vista, $\eta$ parece ser "o tamanho do passo".
   Entender que ele só controla a escala das atualizações *em relação aos pesos iniciais* (item
   1.D) foi o que explicou os resultados das duas execuções.
3. *O limitante do teorema de convergência* é enunciado no espaço aumentado
   $[\mathbf{x}, 1]$, com $(\mathbf{w}, b)$ unitário. Calcular a margem fora desse espaço dá um
   número que não corresponde ao teorema.

In [ ]:
def gerar_classes(media_0, media_1, cov, n=N_PER_CLASS):
    """Amostra duas classes de uma normal multivariada 2D com o `rng` global.

    As amostras ficam em blocos — as `n` da Classe 0 primeiro, depois as `n` da
    Classe 1 — e o treino percorre o dataset nessa ordem fixa em toda época.
    Retorna (X, y) com X de shape (2n, 2) e y de shape (2n,), y em {0, 1}.
    """
    X0 = rng.multivariate_normal(media_0, cov, size=n)
    X1 = rng.multivariate_normal(media_1, cov, size=n)
    X = np.vstack([X0, X1])
    y = np.repeat([0, 1], n)
    return X, y


def plotar_dados(X, y, ax, tamanho=8, alpha=0.5):
    """Scatter das duas classes, uma cor por classe."""
    for k in (0, 1):
        ax.scatter(X[y == k, 0], X[y == k, 1], s=tamanho, alpha=alpha,
                   color=CORES[k], label=ROTULOS[k])
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")


def plotar_fronteira(w, b, ax, **kw):
    """Desenha a reta w·x + b = 0 dentro dos limites atuais do eixo."""
    x1 = np.array(ax.get_xlim())
    if abs(w[1]) > 1e-12:                       # x2 = -(w1·x1 + b) / w2
        ax.plot(x1, -(w[0] * x1 + b) / w[1], **kw)
    else:                                       # reta vertical
        ax.axvline(-b / w[0], **kw)

In [ ]:
def step(z):
    """Função degrau: 1 se z >= 0, senão 0."""
    return (z >= 0).astype(int)


def predict(X, w, b):
    """Predição do perceptron: y_hat = step(w·x + b), vetorizada sobre as linhas de X."""
    return step(X @ w + b)


def acuracia(X, y, w, b):
    """Fração de amostras de todo o dataset classificadas corretamente."""
    return float((predict(X, w, b) == y).mean())


def treinar_perceptron(X, y, w_init, eta=0.01, max_epocas=100):
    """Treina um perceptron de camada única com a regra de erro para rótulos {0, 1}.

    Para cada amostra (x, y), na ordem do dataset:
        y_hat = step(w·x + b)
        w <- w + eta·(y - y_hat)·x
        b <- b + eta·(y - y_hat)
    O erro (y - y_hat) é 0 num acerto (nenhuma atualização) e ±1 num erro.

    Para quando uma época inteira não produz nenhuma atualização, ou após
    `max_epocas`. O bias começa em 0; `w_init` vem de fora para que dois
    treinos possam partir exatamente do mesmo ponto.

    Pocket: sempre que uma atualização leva a uma acurácia (no dataset inteiro)
    maior que qualquer outra já vista, (w, b) é copiado para o "bolso". É a
    única adição ao laço; no Exercício 1 ela não altera em nada o treino.
    """
    w = np.array(w_init, dtype=float).copy()
    b = 0.0

    # o bolso começa com os pesos iniciais (época 0)
    pocket = {"w": w.copy(), "b": b, "acc": acuracia(X, y, w, b), "epoca": 0}

    hist = []                                   # uma linha por época
    for epoca in range(1, max_epocas + 1):
        n_updates = 0
        for x_i, y_i in zip(X, y):
            y_hat = 1 if x_i @ w + b >= 0 else 0
            erro = y_i - y_hat                  # 0, +1 ou -1
            if erro != 0:
                w = w + eta * erro * x_i
                b = b + eta * erro
                n_updates += 1
                # pocket: guarda o melhor (w, b) visto até agora
                acc_atual = acuracia(X, y, w, b)
                if acc_atual > pocket["acc"]:
                    pocket = {"w": w.copy(), "b": b, "acc": acc_atual, "epoca": epoca}

        hist.append({
            "epoca": epoca,
            "updates": n_updates,
            "acuracia": acuracia(X, y, w, b),   # pesos correntes, fim da época
            "acuracia_pocket": pocket["acc"],   # melhor até aqui
            "w1": w[0], "w2": w[1], "b": b,
        })
        if n_updates == 0:                      # passada limpa -> convergiu
            break

    return {"w": w, "b": b, "epocas": epoca, "convergiu": n_updates == 0,
            "hist": pd.DataFrame(hist).set_index("epoca"), "pocket": pocket}

# Exercício 1 — Dados Separáveis: o caso para o qual o perceptron foi projetado

## A — Gerar os dados

Duas classes 2D, 1000 amostras cada, de normais multivariadas:

| Classe | Média | Covariância |
|:------:|:-----:|:-----------:|
| 0 | $[1{,}5,\ 1{,}5]$ | $\begin{bmatrix}0{,}5 & 0\\ 0 & 0{,}5\end{bmatrix}$ |
| 1 | $[5,\ 5]$ | $\begin{bmatrix}0{,}5 & 0\\ 0 & 0{,}5\end{bmatrix}$ |

In [ ]:
COV_1 = np.array([[0.5, 0.0], [0.0, 0.5]])
X1, y1 = gerar_classes([1.5, 1.5], [5.0, 5.0], COV_1)   # 1º e 2º sorteios do rng

print(f"X1.shape = {X1.shape}   y1.shape = {y1.shape}")
print("amostras por classe:", np.bincount(y1))
pd.DataFrame(X1, columns=["x1", "x2"]).assign(classe=y1).groupby("classe").agg(
    media_x1=("x1", "mean"), media_x2=("x2", "mean"),
    var_x1=("x1", "var"), var_x2=("x2", "var"),
).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plotar_dados(X1, y1, ax)
ax.set_title("Figura 1 — Exercício 1: dados separáveis (1000 amostras por classe)")
ax.legend(title="Classe", markerscale=2)
ax.set_aspect("equal")
plt.show()

As médias empíricas, $(1{,}450,\ 1{,}473)$ e $(5{,}010,\ 5{,}013)$, e as variâncias, entre
$0{,}491$ e $0{,}513$, reproduzem os parâmetros. Os centros distam
$\lVert[3{,}5,\ 3{,}5]\rVert \approx 4{,}95$, cerca de 7 desvios-padrão
($\sigma = \sqrt{0{,}5} \approx 0{,}71$). A Figura 1 mostra duas nuvens compactas, com uma faixa
vazia entre elas.

## B — Implementar o perceptron

A implementação está nas células de código do início da página (`step`, `predict`, `acuracia` e
`treinar_perceptron`), comentada, e segue exatamente a especificação:

- **Predição:** $\hat{y} = \text{step}(\mathbf{w} \cdot \mathbf{x} + b)$, com
  $\text{step}(z) = 1$ se $z \geq 0$ e $0$ caso contrário.
- **Regra de atualização**, para rótulos $y, \hat{y} \in \{0, 1\}$:
  $$\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat{y})\,\mathbf{x}, \qquad
    b \leftarrow b + \eta\,(y - \hat{y})$$
  O erro $(y - \hat{y})$ vale $0$ num acerto, e nesse caso não há atualização. Vale $+1$ num falso
  negativo (soma $\eta\mathbf{x}$, empurrando $\mathbf{w}$ *em direção* ao ponto) e $-1$ num falso
  positivo (subtrai $\eta\mathbf{x}$). A forma $\mathbf{w} + \eta\,y\,\mathbf{x}$ dos livros que
  usam rótulos $\pm 1$ nunca atualizaria na Classe 0, porque $y = 0$, e jamais corrigiria um falso
  positivo.
- **Inicialização:** $\mathbf{w} \sim$ `rng.normal(0, 0.01, size=2)`, $b = 0$. Nunca
  $\mathbf{w} = \mathbf{0}$ (o motivo está no item D.3).
- **Taxa de aprendizado:** $\eta = 0{,}01$.
- **Parada:** quando uma passada inteira não produz nenhuma atualização, ou após 100 épocas. A
  acurácia no dataset completo é registrada ao fim de cada época.

## C — Treinar e medir

In [ ]:
# Pesos iniciais sorteados UMA vez e reaproveitados nos dois treinos do item D,
# para que a única diferença entre eles seja o eta.
w0_ex1 = rng.normal(0, 0.01, size=2)                   # 3º sorteio do rng
print("w inicial =", w0_ex1, "   b inicial = 0")

res1 = treinar_perceptron(X1, y1, w0_ex1, eta=0.01)

print(f"\nconvergiu: {res1['convergiu']}  em {res1['epocas']} épocas")
print(f"w final = [{res1['w'][0]:.6f}, {res1['w'][1]:.6f}]")
print(f"b final = {res1['b']:.4f}")
print(f"acurácia final = {acuracia(X1, y1, res1['w'], res1['b'])*100:.2f}%")
print(f"total de atualizações = {res1['hist']['updates'].sum()}")
print("atualizações por época:", res1["hist"]["updates"].tolist())

In [ ]:
erro1 = predict(X1, res1["w"], res1["b"]) != y1

fig, ax = plt.subplots(figsize=(7, 6))
plotar_dados(X1, y1, ax)
ax.set_aspect("equal")
ax.set_xlim(ax.get_xlim()); ax.set_ylim(ax.get_ylim())
plotar_fronteira(res1["w"], res1["b"], ax, color="k", lw=2,
                 label=r"fronteira $\mathbf{w}\cdot\mathbf{x}+b=0$")
ax.scatter(X1[erro1, 0], X1[erro1, 1], s=90, facecolors="none", edgecolors="k",
           lw=1.5, label=f"mal classificados ({erro1.sum()})")
ax.set_title(r"Figura 2 — Exercício 1: fronteira de decisão ($\eta = 0{,}01$)")
ax.legend(markerscale=1.5, loc="upper left")
plt.show()

In [ ]:
h1 = res1["hist"]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(h1.index, h1["acuracia"] * 100, marker="o", ms=4, lw=1.8, color="#2ca02c",
        label=r"acurácia (pesos correntes, $\eta = 0{,}01$)")
ax.set_xlabel("época")
ax.set_ylabel("acurácia no dataset completo (%)")
ax.set_title("Figura 3 — Exercício 1: acurácia × época")
ax.set_xticks(h1.index[::2] if len(h1) > 15 else h1.index)
ax.legend(loc="lower right")
plt.show()

print(h1[["updates", "acuracia"]].T.round(4).to_string())

**Resultados ($\eta = 0{,}01$), partindo de $\mathbf{w}_0 = [0{,}002532,\ 0{,}008952]$, $b_0 = 0$:**

| Quantidade | Valor |
|:---|:---|
| $\mathbf{w}$ final | $[0{,}050497,\ 0{,}028872]$ |
| $b$ final | $-0{,}25$ |
| épocas | **26**: houve atualizações até a época 25, e a época 26 é a passada limpa |
| total de atualizações | 73 |
| acurácia final | **100,00%** (2000/2000, nenhum ponto mal classificado na Figura 2) |

A fronteira final é $0{,}050497\,x_1 + 0{,}028872\,x_2 - 0{,}25 = 0$. Ela atravessa a faixa vazia
entre as nuvens, a $-b/\lVert\mathbf{w}\rVert = 4{,}298$ da origem. Não é a mediatriz das médias:
é inclinada em relação à diagonal, pois sua normal $\mathbf{w}/\lVert\mathbf{w}\rVert = [0{,}868,\ 0{,}496]$
não é $[0{,}707,\ 0{,}707]$. O perceptron para na **primeira** reta que separa os dados, não
na melhor.

**Leitura da Figura 3.** A curva não sobe de forma monótona. Ela começa em 50,3%, chega a 85,55%
na época 8, cai para 60,05% na época 9, e oscila assim até fixar-se em 100% na época 25. Isso
vem da ordem em blocos. Os erros da Classe 0 só aparecem na primeira metade da época, e os da
Classe 1 só na segunda. A acurácia é medida logo depois do bloco da Classe 1, cujas correções
empurram a fronteira para o lado da Classe 0. Quando esse empurrão passa do ponto, parte da
Classe 0 fica do lado errado, e a acurácia de fim de época cai. Quando as duas correções se
equilibram, a fronteira cai na faixa vazia, e nenhuma das duas metades da época gera erros.

## D — Análise

### D.1 — Por que dados separáveis convergem rápido?

In [ ]:
# Margem geométrica de cada ponto em relação à fronteira final (distância com sinal,
# positiva quando o ponto está do lado certo) e raio R do dataset aumentado [x, 1].
s1 = 2 * y1 - 1                                        # rótulos em {-1, +1}
dist1 = s1 * (X1 @ res1["w"] + res1["b"]) / np.linalg.norm(res1["w"])
R1 = np.linalg.norm(np.hstack([X1, np.ones((len(X1), 1))]), axis=1).max()
print(f"menor distância de um ponto à fronteira final: {dist1.min():.4f}")
print(f"pontos a menos de 0,1 da fronteira: {(dist1 < 0.1).sum()}")
print(f"pontos a mais de 1,0 da fronteira:  {(dist1 > 1.0).sum()} de {len(X1)}")
print(f"R = max ||[x, 1]|| = {R1:.3f}")

# Margem gamma de um bom separador conhecido — a família x1 + x2 = c, perpendicular à
# reta que liga as médias —, medida no espaço aumentado [x, 1] com (w, b) unitário,
# que é a forma em que o teorema de convergência é enunciado.
X1_aum = np.hstack([X1, np.ones((len(X1), 1))])
def margem_aumentada(c):
    u = np.array([1.0, 1.0, -c])
    u = u / np.linalg.norm(u)                   # (w, b) unitário
    return np.min(s1 * (X1_aum @ u))            # pior ponto define a margem

gamma, c_best = max((margem_aumentada(c), c) for c in np.linspace(5, 8, 3001))
print(f"melhor separador x1 + x2 = {c_best:.3f}:  gamma = {gamma:.4f}")
print(f"limitante de Novikoff (R / gamma)^2 = {(R1 / gamma)**2:,.0f} atualizações"
      f"   (observado: {res1['hist']['updates'].sum()})")

**A regra só atualiza em erro.** Um ponto classificado corretamente gera $(y - \hat{y}) = 0$ e
não mexe nos pesos. Nestes dados, quase todo ponto está muito longe de qualquer fronteira
razoável: **1956 dos 2000** ficam a mais de 1,0 da fronteira final, e só 3 ficam a menos de 0,1.
Os pontos centrais das nuvens nunca pesam no treino. A única coisa que move os pesos é o punhado
de pontos que, numa dada época, cai do lado errado.

**O número de atualizações por época.** Foi de 2 a 4 desde o início, caiu para 1 na época 25 e
para **0** na época 26, num total de **73** atualizações. Ele não começa alto por causa da ordem
em blocos: dentro de um bloco, uma ou duas correções já levam a fronteira para o lado certo de
*todo* o bloco. Cada correção desloca $\mathbf{w}\cdot\mathbf{x}$ de $\eta\lVert\mathbf{x}\rVert^2$,
e os pontos de uma mesma classe estão próximos, então ninguém mais no bloco erra. O que muda com
o treino é se as correções de uma metade da época desfazem as da outra. Enquanto isso acontece,
as duas metades seguem gerando erros. Quando a fronteira cai dentro da faixa vazia, os dois
blocos ficam inteiramente corretos de uma vez. Uma fronteira assim existe, e a convergência é
garantida.

**A garantia formal** é o teorema de convergência do perceptron (Novikoff). Se existe
$(\mathbf{w}^*, b^*)$ unitário que classifica todos os pontos com margem $\gamma > 0$ no espaço
aumentado $[\mathbf{x}, 1]$, e $R = \max\lVert[\mathbf{x}, 1]\rVert$, então o número **total**
de atualizações é no máximo $(R/\gamma)^2$, independentemente de $\eta$ e da ordem das amostras.
Aqui, a reta $x_1 + x_2 = 6{,}745$ tem $\gamma = 0{,}0396$ e $R = 9{,}174$, o que dá um
limitante de **53.753** atualizações. As 73 observadas ficam muito abaixo dele. O limitante é
folgado, mas o essencial é que ele é **finito**: o total de atualizações é limitado, então o
número de atualizações por época tem que chegar a zero.

### D.2 — Repetir com $\eta = 1{,}0$

Mesmo dataset, mesmo $\mathbf{w}_0$, mesmo $b_0 = 0$, mesma ordem. Só o $\eta$ muda.

In [ ]:
res1_eta1 = treinar_perceptron(X1, y1, w0_ex1, eta=1.0)   # mesmo w inicial

def direcao(w):
    return w / np.linalg.norm(w)

comp = pd.DataFrame({
    "eta = 0,01": [res1["epocas"], res1["hist"]["updates"].sum(),
                   acuracia(X1, y1, res1["w"], res1["b"]),
                   res1["w"].round(4), round(res1["b"], 4),
                   direcao(res1["w"]).round(6),
                   -res1["b"] / np.linalg.norm(res1["w"])],
    "eta = 1,0": [res1_eta1["epocas"], res1_eta1["hist"]["updates"].sum(),
                  acuracia(X1, y1, res1_eta1["w"], res1_eta1["b"]),
                  res1_eta1["w"].round(4), round(res1_eta1["b"], 4),
                  direcao(res1_eta1["w"]).round(6),
                  -res1_eta1["b"] / np.linalg.norm(res1_eta1["w"])],
}, index=["épocas", "atualizações", "acurácia final", "w", "b", "w / ||w||",
          "distância da fronteira à origem  -b/||w||"])

ang = np.degrees(np.arccos(np.clip(direcao(res1["w"]) @ direcao(res1_eta1["w"]), -1, 1)))
print(f"ângulo entre as direções de w: {ang:.4f} graus")
print(f"||w0|| / ||eta·x|| médio:  eta=0,01 -> "
      f"{np.linalg.norm(w0_ex1) / (0.01 * np.linalg.norm(X1, axis=1).mean()):.3f}   "
      f"eta=1,0 -> {np.linalg.norm(w0_ex1) / (1.0 * np.linalg.norm(X1, axis=1).mean()):.5f}")
comp

| | $\eta = 0{,}01$ | $\eta = 1{,}0$ |
|:---|:---:|:---:|
| épocas | **26** | **37** |
| atualizações | 73 | 101 |
| acurácia final | **100,00%** | **100,00%** |
| $\mathbf{w}$ final | $[0{,}0505,\ 0{,}0289]$ | $[5{,}8706,\ 3{,}3592]$ |
| $b$ final | $-0{,}25$ | $-31{,}0$ |
| $\mathbf{w}/\lVert\mathbf{w}\rVert$ | $[0{,}868123,\ 0{,}496349]$ | $[0{,}867950,\ 0{,}496652]$ |
| distância da fronteira à origem, $-b/\lVert\mathbf{w}\rVert$ | 4,298 | 4,583 |

As duas execuções chegam a 100%, **com fronteiras diferentes**. Neste caso, a diferença está quase
toda na **posição** da reta, e não na direção. As normais formam um ângulo de apenas 0,02°, então as
duas fronteiras são praticamente paralelas, mas a de $\eta = 1{,}0$ está 0,285 mais longe da
origem, mais perto da Classe 1. Por outro caminho, a execução com $\eta = 1{,}0$ levou 11 épocas
e 28 atualizações a mais.

**O que $\eta$ controla.** A fronteira $\{\mathbf{x} : \mathbf{w}\cdot\mathbf{x} + b = 0\}$ não muda
se $(\mathbf{w}, b)$ for multiplicado por qualquer constante positiva. Por isso o "tamanho do passo"
em si não tem significado geométrico. O que $\eta$ controla é o **peso relativo da inicialização**.
Cada atualização soma $\pm\eta\mathbf{x}$, e $\lVert\mathbf{w}_0\rVert \approx 0{,}0093$:

- Com $\eta = 0{,}01$, uma atualização típica tem norma $\approx 0{,}01 \cdot \lVert\mathbf{x}\rVert$,
  e $\lVert\mathbf{w}_0\rVert$ é **20%** dela. A inicialização pesa nas primeiras decisões, desvia a
  trajetória e leva a outra fronteira, alcançada em 26 épocas.
- Com $\eta = 1{,}0$, $\lVert\mathbf{w}_0\rVert$ é **0,2%** de uma atualização. Já depois da primeira
  correção, a inicialização é ruído desprezível, e a execução é, na prática, a de partida em zero.
  O item D.3 confirma isso numericamente: partindo de $\mathbf{w} = \mathbf{0}$, o treino também leva
  37 épocas e termina em $\mathbf{w} = [5{,}8681,\ 3{,}3503]$, $b = -31$, quase idêntico ao
  $[5{,}8706,\ 3{,}3592]$, $b = -31$ acima.

Ou seja, $\eta$ grande **apaga** a inicialização, e $\eta$ pequeno a **preserva**. A trajetória
depende de $\eta$ só através da razão $\lVert\mathbf{w}_0\rVert / \eta$.

### D.3 — E se partíssemos de $\mathbf{w} = \mathbf{0}$, $b = 0$?

In [ ]:
# Verificação numérica da prova algébrica: a partir de w = 0, b = 0.
# (np.zeros não consome o rng.)
z_a = treinar_perceptron(X1, y1, np.zeros(2), eta=0.01)
z_b = treinar_perceptron(X1, y1, np.zeros(2), eta=1.0)

print(f"eta=0,01:  w = {z_a['w']},  b = {z_a['b']:.4f},  épocas = {z_a['epocas']}")
print(f"eta=1,0 :  w = {z_b['w']},  b = {z_b['b']:.4f},  épocas = {z_b['epocas']}")
print(f"razão w (eta=1 / eta=0,01) = {z_b['w'] / z_a['w']}")
print(f"razão b                    = {z_b['b'] / z_a['b']:.6f}")
print("mesma sequência de atualizações por época:",
      z_a["hist"]["updates"].tolist() == z_b["hist"]["updates"].tolist())

**Proposição.** Partindo de $\mathbf{w} = \mathbf{0}$, $b = 0$, dois treinos com $\eta_1$ e $\eta_2$
(ambos $> 0$), sobre os mesmos dados e na mesma ordem, satisfazem, depois de **cada** amostra
processada $t$:
$$\mathbf{w}^{(2)}_t = \frac{\eta_2}{\eta_1}\,\mathbf{w}^{(1)}_t, \qquad
  b^{(2)}_t = \frac{\eta_2}{\eta_1}\,b^{(1)}_t.$$

**Prova (indução em $t$).** Seja $c = \eta_2/\eta_1 > 0$.

- *Base:* $\mathbf{w}^{(1)}_0 = \mathbf{w}^{(2)}_0 = \mathbf{0}$ e $b^{(1)}_0 = b^{(2)}_0 = 0$, e
  $\mathbf{0} = c \cdot \mathbf{0}$.
- *Passo:* suponha $\mathbf{w}^{(2)}_t = c\,\mathbf{w}^{(1)}_t$ e $b^{(2)}_t = c\,b^{(1)}_t$. Para a
  próxima amostra $(\mathbf{x}, y)$:
  $$z^{(2)} = \mathbf{w}^{(2)}_t\cdot\mathbf{x} + b^{(2)}_t = c\,(\mathbf{w}^{(1)}_t\cdot\mathbf{x} + b^{(1)}_t) = c\,z^{(1)}.$$
  Como $c > 0$, $z^{(2)} \geq 0 \iff z^{(1)} \geq 0$, inclusive no caso $z = 0$. Logo
  $\hat{y}^{(2)} = \hat{y}^{(1)}$, e o erro $e = y - \hat{y}$ é **o mesmo** nos dois treinos. Então
  $$\mathbf{w}^{(2)}_{t+1} = c\,\mathbf{w}^{(1)}_t + \eta_2\,e\,\mathbf{x}
    = c\,\big(\mathbf{w}^{(1)}_t + \eta_1\,e\,\mathbf{x}\big) = c\,\mathbf{w}^{(1)}_{t+1},$$
  e da mesma forma $b^{(2)}_{t+1} = c\,b^{(1)}_{t+1}$. $\blacksquare$

Em forma fechada: a partir do zero, $\mathbf{w}_t = \eta\sum_k e_k\,\mathbf{x}_k$ e
$b_t = \eta\sum_k e_k$, e a sequência de erros $e_k$ não depende de $\eta$. **O $\eta$ é um fator
comum que sai da soma.**

**Consequências.** As predições são idênticas em todas as amostras de todas as épocas. Os erros
caem nos mesmos pontos, a passada limpa acontece na mesma época e o **número de épocas é igual**.
A fronteira final $\{c\,(\mathbf{w}\cdot\mathbf{x} + b) = 0\}$ é **a mesma reta**. Partindo do zero,
$\eta$ não tem efeito nenhum sobre o resultado.

**Verificação numérica.** Com $\mathbf{w} = \mathbf{0}$, os dois treinos levam **37 épocas** cada
e a mesma sequência de atualizações por época. A razão entre os pesos finais é exatamente
$[100,\ 100]$ para $\mathbf{w}$ e $100$ para $b$, ou seja, $\eta_2/\eta_1 = 1{,}0/0{,}01$.

**Por que o item B proíbe a partida em zero.** Com $\mathbf{w}_0 \neq \mathbf{0}$, a forma fechada
vira $\mathbf{w}_t = \mathbf{w}_0 + \eta\sum_k e_k\,\mathbf{x}_k$, e $\eta$ não sai mais como fator
comum. É exatamente esse termo $\mathbf{w}_0$ que fez as duas execuções do item D.2 divergirem
(26 contra 37 épocas). Sem ele, a pergunta "o que muda com $\eta$?" teria uma resposta trivial: nada.

# Exercício 2 — Dados Sobrepostos: o caso que o perceptron não resolve

## A — Gerar os dados

| Classe | Média | Covariância |
|:------:|:-----:|:-----------:|
| 0 | $[3,\ 3]$ | $\begin{bmatrix}1{,}5 & 0\\ 0 & 1{,}5\end{bmatrix}$ |
| 1 | $[4,\ 4]$ | $\begin{bmatrix}1{,}5 & 0\\ 0 & 1{,}5\end{bmatrix}$ |

In [ ]:
COV_2 = np.array([[1.5, 0.0], [0.0, 1.5]])
X2, y2 = gerar_classes([3.0, 3.0], [4.0, 4.0], COV_2)   # 4º e 5º sorteios do rng

print(f"X2.shape = {X2.shape}   y2.shape = {y2.shape}")
print("amostras por classe:", np.bincount(y2))
print(f"||x|| médio = {np.linalg.norm(X2, axis=1).mean():.3f}")
pd.DataFrame(X2, columns=["x1", "x2"]).assign(classe=y2).groupby("classe").agg(
    media_x1=("x1", "mean"), media_x2=("x2", "mean"),
    var_x1=("x1", "var"), var_x2=("x2", "var"),
).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plotar_dados(X2, y2, ax, alpha=0.45)
ax.set_title("Figura 4 — Exercício 2: dados sobrepostos (1000 amostras por classe)")
ax.legend(title="Classe", markerscale=2)
ax.set_aspect("equal")
plt.show()

As médias empíricas são $(3{,}032,\ 3{,}001)$ e $(4{,}021,\ 3{,}946)$, e as variâncias ficam entre
$1{,}417$ e $1{,}640$. Agora os centros distam só $\lVert[1,\ 1]\rVert \approx 1{,}41$, cerca de
**1,15 desvio-padrão** ($\sigma = \sqrt{1{,}5} \approx 1{,}22$), contra 7 no Exercício 1. Na
Figura 4 as nuvens se sobrepõem quase por inteiro, e nenhuma reta as separa. A norma média dos
pontos é $\lVert\mathbf{x}\rVert \approx 5{,}11$, valor que será usado no item D.

## B — Treinar guardando os melhores pesos

A mesma função `treinar_perceptron`, **sem alteração**, com o mesmo $\eta = 0{,}01$ e o mesmo limite
de 100 épocas. Como os dados não são separáveis, o laço nunca para de atualizar, e acompanhamos dois
conjuntos de pesos:

- **finais:** o que o laço tem em mãos depois da última época;
- **pocket:** a cópia de $(\mathbf{w}, b)$ feita sempre que uma atualização leva a uma acurácia, no
  dataset inteiro, maior do que qualquer outra já vista.

In [ ]:
# Mesma função, mesmo eta, mesmo limite de 100 épocas; novos pesos iniciais.
w0_ex2 = rng.normal(0, 0.01, size=2)                   # 6º sorteio do rng
print("w inicial =", w0_ex2, "   b inicial = 0")

res2 = treinar_perceptron(X2, y2, w0_ex2, eta=0.01, max_epocas=100)
pk = res2["pocket"]
acc_final2 = acuracia(X2, y2, res2["w"], res2["b"])

print(f"\nconvergiu: {res2['convergiu']}   épocas executadas: {res2['epocas']}")
print(f"FINAL : w = [{res2['w'][0]:.6f}, {res2['w'][1]:.6f}]   b = {res2['b']:.4f}"
      f"   acurácia = {acc_final2*100:.2f}%")
print(f"POCKET: w = [{pk['w'][0]:.6f}, {pk['w'][1]:.6f}]   b = {pk['b']:.4f}"
      f"   acurácia = {pk['acc']*100:.2f}%   (época {pk['epoca']})")

**Resultados**, partindo de $\mathbf{w}_0 = [0{,}012158,\ -0{,}004510]$, $b_0 = 0$. O treino não
convergiu e usou todas as **100 épocas**.

| Pesos | $\mathbf{w}$ | $b$ | acurácia |
|:---|:---:|:---:|:---:|
| **finais** (após a época 100) | $[0{,}054484,\ 0{,}048043]$ | $-0{,}07$ | **50,15%** |
| **pocket** (melhor, na época 86) | $[0{,}010664,\ 0{,}008727]$ | $-0{,}07$ | **71,10%** |

Os dois números estão tão longe um do outro quanto o enunciado previa. Os pesos finais acertam
50,15%, praticamente um chute: eles classificam **99,85%** dos pontos como Classe 1. O pocket
chega a 71,10%. Repare que os dois têm o **mesmo $b = -0{,}07$**: o que os separa é só a norma de
$\mathbf{w}$, 0,0726 nos finais contra 0,0138 no pocket. O item D explica por que isso basta.

## C — Figuras

In [ ]:
pred_final2 = predict(X2, res2["w"], res2["b"])
pred_pk2 = predict(X2, pk["w"], pk["b"])
erro_final2 = pred_final2 != y2
erro_pk2 = pred_pk2 != y2

NOME_PESOS = {"final": "finais", "pocket": "pocket"}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for ax, (nome, w_, b_, err, cor) in zip(axes, [
    ("final", res2["w"], res2["b"], erro_final2, "k"),
    ("pocket", pk["w"], pk["b"], erro_pk2, "#ff7f0e"),
]):
    plotar_dados(X2, y2, ax, tamanho=6, alpha=0.35)
    ax.scatter(X2[err, 0], X2[err, 1], marker="x", s=14, lw=0.8, color="0.25",
               label=f"mal classificados pelos pesos {NOME_PESOS[nome]} ({err.sum()})")
    ax.set_xlim(-2.5, 9.5); ax.set_ylim(-2.5, 9.5); ax.set_aspect("equal")
    # as duas fronteiras nos dois painéis; a do painel em destaque, mais grossa
    plotar_fronteira(res2["w"], res2["b"], ax, color="k",
                     lw=2.5 if nome == "final" else 1.2,
                     ls="-" if nome == "final" else "--",
                     label=f"fronteira final ({acc_final2*100:.2f}%)")
    plotar_fronteira(pk["w"], pk["b"], ax, color="#ff7f0e",
                     lw=2.5 if nome == "pocket" else 1.2,
                     ls="-" if nome == "pocket" else "--",
                     label=f"fronteira pocket ({pk['acc']*100:.2f}%)")
    ax.set_title(f"pontos marcados: erros dos pesos {NOME_PESOS[nome]}")
    ax.legend(loc="upper left", fontsize=8, markerscale=1.5)
fig.suptitle("Figura 5 — Exercício 2: fronteiras final e pocket", y=1.0, fontsize=12)
fig.tight_layout()
plt.show()

print(f"pesos finais: predizem Classe 1 para {pred_final2.mean()*100:.2f}% dos pontos")
print(f"pesos pocket: predizem Classe 1 para {pred_pk2.mean()*100:.2f}% dos pontos")

In [ ]:
h2 = res2["hist"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(h2.index, h2["acuracia"] * 100, lw=1.3, color="k",
        label="acurácia dos pesos correntes (fim da época)")
ax.plot(h2.index, h2["acuracia_pocket"] * 100, lw=2.2, color="#ff7f0e",
        label="melhor até agora (pocket)")
ax.axhline(50, color="0.5", ls=":", lw=1, label="chute (50%)")
ax.set_xlabel("época")
ax.set_ylabel("acurácia no dataset completo (%)")
ax.set_ylim(45, 80)
ax.set_title("Figura 6 — Exercício 2: acurácia × época")
ax.legend(loc="center right")
plt.show()

print(f"acurácia corrente ao fim das épocas: mín {h2['acuracia'].min()*100:.2f}%  "
      f"máx {h2['acuracia'].max()*100:.2f}%  média {h2['acuracia'].mean()*100:.2f}%")
print(f"atualizações por época: mín {h2['updates'].min()}  máx {h2['updates'].max()}"
      f"  média {h2['updates'].mean():.2f}")

**Leitura das Figuras 5 e 6.** Na Figura 5 a fronteira final (preta) passa *abaixo e à esquerda*
de toda a nuvem e corta a diagonal $x_1 = x_2$ em $(0{,}68,\ 0{,}68)$. Quase todos os pontos
ficam do lado "Classe 1", e os erros são justamente os 997 pontos da Classe 0 que ela não
separa. A fronteira pocket (laranja) atravessa o meio da nuvem e corta a diagonal em
$(3{,}61,\ 3{,}61)$, perto do centro dos dados, $(3{,}53,\ 3{,}47)$. Ela é quase perpendicular à
direção que liga as médias: sua normal forma 5,7° com a diagonal. Ela erra 578 pontos (245 da Classe 0 e 333 da Classe 1), os da zona de sobreposição, que nenhuma reta acertaria.

Na Figura 6, a acurácia dos pesos correntes ao fim de cada época fica presa entre **50,05% e
51,15%** (média 50,31%) nas 100 épocas. A curva do pocket sobe em degraus e fica em **71,10%**
a partir da época 86.

## D — Análise

### D.1 — Por que os pesos finais ficam em ~50% e o pocket em ~71%?

**Referência.** O enunciado fala em "cerca de 73%" para a melhor reta. Como as duas classes têm a
mesma covariância isotrópica, a melhor reta é a mediatriz das médias, $x_1 + x_2 = 7$, com
acurácia teórica $\Phi(d/2)$, onde $d = \lVert\mu_1 - \mu_0\rVert/\sigma = 1{,}155$. Isso dá
**71,81%** na população e **71,25%** nesta amostra (calculado na última célula do item D.2). O
pocket, com **71,10%**, fica a 0,15 ponto percentual da mediatriz.

In [ ]:
# Onde a fronteira final corta a diagonal x1 = x2 (a direção que liga as médias)?
# Na diagonal, w·x + b = 0  ->  t·(w1 + w2) + b = 0  ->  t = -b / (w1 + w2).
def corte_diagonal(w, b):
    return -b / (w[0] + w[1])

print(f"centro dos dados: {X2.mean(axis=0).round(3)}")
print(f"fronteira final  corta a diagonal em ({corte_diagonal(res2['w'], res2['b']):.3f}, "
      f"{corte_diagonal(res2['w'], res2['b']):.3f})")
print(f"fronteira pocket corta a diagonal em ({corte_diagonal(pk['w'], pk['b']):.3f}, "
      f"{corte_diagonal(pk['w'], pk['b']):.3f})")
print(f"distância da fronteira final à origem: "
      f"{abs(res2['b']) / np.linalg.norm(res2['w']):.3f}")
print(f"\n||w|| final = {np.linalg.norm(res2['w']):.4f}   b final = {res2['b']:.4f}")
print(f"|b| / ||w|| necessário para a fronteira passar pelo centro: "
      f"{abs(X2.mean(axis=0) @ direcao(res2['w'])):.3f}")
print(f"faixa de b ao longo das 100 épocas: [{h2['b'].min():.2f}, {h2['b'].max():.2f}]")
print(f"por erro: |Δb| = eta = 0,01   ||Δw|| = eta·||x|| ≈ "
      f"{0.01 * np.linalg.norm(X2, axis=1).mean():.4f}")

In [ ]:
# Dentro de uma época: a última época, amostra a amostra, para ver o ciclo.
# Repete a época 100 a partir dos pesos do fim da época 99 (sem sortear nada).
w_c = np.array([h2.loc[99, "w1"], h2.loc[99, "w2"]]); b_c = h2.loc[99, "b"]
eventos = []
for i, (x_i, y_i) in enumerate(zip(X2, y2)):
    erro = y_i - (1 if x_i @ w_c + b_c >= 0 else 0)
    if erro != 0:
        w_c = w_c + 0.01 * erro * x_i
        b_c = b_c + 0.01 * erro
        eventos.append({"amostra": i, "classe": y_i, "erro": erro,
                        "w1": w_c[0], "w2": w_c[1], "b": b_c,
                        "acurácia após": acuracia(X2, y2, w_c, b_c),
                        "% predito como 1": predict(X2, w_c, b_c).mean() * 100})
assert np.allclose(w_c, res2["w"]) and np.isclose(b_c, res2["b"])
pd.DataFrame(eventos).round(4)

**Onde a fronteira final está.** A fronteira passa pelo centro da nuvem só se
$|b|/\lVert\mathbf{w}\rVert \approx 4{,}94$, a projeção do centro dos dados na direção de
$\mathbf{w}$. Com $\lVert\mathbf{w}\rVert = 0{,}0726$, seria preciso $b \approx -0{,}36$, mas o
treino deixou $b = -0{,}07$. A reta fica então a só $0{,}96$ da origem, longe da nuvem, com tudo
do mesmo lado.

**Por que o laço a deixa ali. A pista do enunciado: $b$ anda devagar.** A cada erro,
$$|\Delta b| = \eta = 0{,}01, \qquad \lVert\Delta\mathbf{w}\rVert = \eta\,\lVert\mathbf{x}\rVert \approx 0{,}0511,$$
ou seja, $\mathbf{w}$ anda **cerca de 5 vezes mais** que $b$ a cada correção. Como
$\mathbf{x} \approx (3{,}5,\ 3{,}5)$ em todos os pontos, o efeito de uma correção sobre
$\mathbf{w}\cdot\mathbf{x} + b$ é quase o mesmo para o dataset inteiro:
$\eta\,(\lVert\mathbf{x}\rVert^2 + 1) \approx 0{,}27$. Uma única correção não move a fronteira para
dentro da nuvem. Ela **passa a nuvem inteira para o outro lado**. A tabela da célula acima mostra
a época 100 amostra por amostra, e ela tem só **duas** atualizações:

1. **Amostra 0 (Classe 0), predita como 1.** $\mathbf{w} \leftarrow \mathbf{w} - \eta\mathbf{x}$
   cai para $[0{,}0039,\ 0{,}0006]$, quase zero, e $b = -0{,}08$. Agora
   $\mathbf{w}\cdot\mathbf{x} + b < 0$ em todo ponto: **0%** dos pontos preditos como 1 e acurácia
   de 50,00%. O resto do bloco da Classe 0 está "correto", e não há mais atualizações.
2. **Amostra 1000 (Classe 1), predita como 0.** $\mathbf{w} \leftarrow \mathbf{w} + \eta\mathbf{x}$
   volta a $[0{,}0545,\ 0{,}0480]$, e $b = -0{,}07$. Agora $\mathbf{w}\cdot\mathbf{x} + b > 0$ em
   quase todo ponto: **99,85%** dos pontos preditos como 1 e acurácia de 50,15%. O resto do bloco
   da Classe 1 está "correto".

Cada época tem uma correção num sentido e outra no sentido oposto, então $b$ sobe e desce e não
acumula: nas 100 épocas ele ficou entre $-0{,}08$ e $-0{,}01$. Para uma reta atravessar a nuvem,
$b$ teria que se afastar de zero muito mais rápido que $\mathbf{w}$ cresce, e a regra faz o
contrário. A fronteira fica perto da origem, e o que sobra para o laço é apenas escolher de que
lado a nuvem inteira fica. Como a época termina sempre com o bloco da Classe 1, o estado ao fim
da época é sempre "quase tudo é Classe 1", e a acurácia fica sempre em ~50%.

**Por que o pocket acerta 71%.** Entre as duas correções, logo depois de um
$\mathbf{w} \leftarrow \mathbf{w} - \eta\mathbf{x}$, há instantes em que $\mathbf{w}$ fica *quase
cancelado*: a subtração tira de $\mathbf{w}$ quase tudo o que a soma anterior pôs, e
$\lVert\mathbf{w}\rVert$ cai para a ordem de $|b|/5$. Nesse instante, $|b|/\lVert\mathbf{w}\rVert$
passa por $\approx 5$, a fronteira atravessa o centro da nuvem e a acurácia salta. O pocket,
com $\lVert\mathbf{w}\rVert = 0{,}0138$ e $b = -0{,}07$, tem $|b|/\lVert\mathbf{w}\rVert = 5{,}08$.
O laço passa por essa configuração só por acaso e logo sai dela. O pocket é o mecanismo que
**guarda** esse instante. A diferença entre 50% e 71% não está no algoritmo de aprendizado, e
sim em *qual* iterado é reportado.

### D.2 — Figura 3 × Figura 6, e o teorema de convergência

In [ ]:
# Experimento complementar (fora do pedido): mesmo treino, mas com as amostras em
# ordem embaralhada. Fica por último para não alterar nenhum sorteio anterior do rng.
perm = rng.permutation(len(X2))                        # 7º sorteio do rng
res2_emb = treinar_perceptron(X2[perm], y2[perm], w0_ex2, eta=0.01)
h2e = res2_emb["hist"]
print(f"FINAL : acurácia = {acuracia(X2, y2, res2_emb['w'], res2_emb['b'])*100:.2f}%")
print(f"POCKET: acurácia = {res2_emb['pocket']['acc']*100:.2f}%  "
      f"(época {res2_emb['pocket']['epoca']})")
print(f"acurácia corrente ao fim das épocas: mín {h2e['acuracia'].min()*100:.2f}%  "
      f"máx {h2e['acuracia'].max()*100:.2f}%")
print(f"atualizações por época: média {h2e['updates'].mean():.1f}")

# Teto teórico: classes com mesma covariância isotrópica -> a melhor reta é a
# mediatriz das médias, e a acurácia é Phi(d / 2) com d = ||mu1 - mu0|| / sigma.
from math import erf, sqrt
d = np.linalg.norm([1.0, 1.0]) / np.sqrt(1.5)
print(f"\nteto teórico da melhor reta: Phi({d/2:.4f}) = {0.5 * (1 + erf(d / 2 / sqrt(2))) * 100:.2f}%")
mediatriz_w, mediatriz_b = np.array([1.0, 1.0]), -7.0       # x1 + x2 = 7
print(f"mediatriz x1 + x2 = 7 nesta amostra: {acuracia(X2, y2, mediatriz_w, mediatriz_b)*100:.2f}%")

**As curvas.** Na Figura 3, a acurácia oscila nas primeiras épocas mas **se fixa** em 100% na época
25, e a época 26 não tem nenhuma atualização. Na Figura 6 ela **nunca se fixa**: as 100 épocas têm
de 2 a 5 atualizações cada, e a acurácia de fim de época oscila entre 50,05% e 51,15% sem
tendência. Só a curva do pocket é monótona, porque ela só guarda máximos.

**O que o teorema garante e qual hipótese é violada.** O teorema de convergência do perceptron
(Rosenblatt/Novikoff, item 1.D.1) garante o seguinte: **se** existe um hiperplano que classifica
todos os pontos corretamente com margem $\gamma > 0$, então o perceptron faz no máximo
$(R/\gamma)^2$ atualizações e para numa passada limpa, com qualquer $\eta$ e qualquer ordem. A
hipótese é a **separabilidade linear**, e este dataset a viola. Nenhuma reta classifica todos os
pontos, e a melhor erra ~28% deles. Não existe $\gamma > 0$, e o teorema não diz nada. O que
continua valendo é o *teorema da ciclagem do perceptron*: os pesos ficam **limitados** mas
**não convergem**. Isso aparece nos dados: ao longo das 100 épocas, $\mathbf{w}$ vagou com
componentes entre 0,012 e 0,069 e $b$ entre $-0{,}08$ e $-0{,}01$, sem se fixar.

**Mais épocas resolvem? Não.** A regra só para quando uma época inteira não tem erros. Como
*todo* $(\mathbf{w}, b)$ erra pelo menos ~28% dos pontos, toda época tem pelo menos uma
atualização. O critério de parada nunca dispara, e o laço não tem ponto fixo para onde ir. Mais
épocas só repetem o mesmo ciclo de duas correções que cancelam uma à outra. Além disso, o estado
ao fim de cada época é sempre o deixado pelas últimas correções, e com esta ordem as últimas são
sempre da Classe 1. A época 1000 terminaria nos mesmos ~50% que a época 100.

**Um $\eta$ menor resolve? Também não, e a regra mostra por quê.** O argumento do item 1.D.3 vale
aqui: $\mathbf{w}_t = \mathbf{w}_0 + \eta\sum_k e_k\mathbf{x}_k$ e $b_t = \eta\sum_k e_k$. Quando
$\eta$ é grande em relação a $\lVert\mathbf{w}_0\rVert$, reduzir $\eta$ só reescala $\mathbf{w}$ e
$b$ **juntos**. A razão $|b|/\lVert\mathbf{w}\rVert$, que é o que posiciona a fronteira, não muda,
e a sequência de erros e o ciclo são os mesmos. Quando $\eta$ fica pequeno o bastante para
$\mathbf{w}_0$ pesar, o que muda é o ponto de partida, mas nenhum ponto de partida cria uma reta
sem erros onde ela não existe. A intuição "passos menores oscilam menos" falha porque não há um
mínimo para onde convergir. O perceptron não desce o gradiente de uma função de custo suave: ele
reage a cada erro, e sempre haverá erros. O que resolve é mudar o **critério**, guardando o melhor
iterado (pocket) ou otimizando uma perda contínua (logística, hinge), e não ajustar $\eta$ nem o
número de épocas.

**Experimento complementar: a ordem das amostras.** O próximo resultado reforça que o iterado
final não significa nada aqui. Treinando *exatamente* igual, só que com as amostras embaralhadas
(última célula, para não alterar nenhum sorteio anterior), os números mudam de natureza:

| | ordem em blocos (relatório) | ordem embaralhada |
|:---|:---:|:---:|
| atualizações por época | 2 a 5 | ~768 |
| acurácia dos pesos finais | 50,15% | 63,95% |
| acurácia de fim de época (faixa) | 50,05% a 51,15% | 61,35% a 70,10% |
| acurácia do pocket | 71,10% (época 86) | 71,85% (época 99) |

Embaralhando, as classes se alternam o tempo todo e a fronteira passa a "tremer" *dentro* da
nuvem em vez de saltar de um lado para o outro. O iterado final melhora, mas continua oscilando
quase 9 pontos percentuais de uma época para a outra, e o treino continua sem convergir: são
centenas de atualizações por época, até a última. Nas duas ordens, o pocket fica em ~71%, no
teto de qualquer reta. **O resultado do pocket é estável; o do iterado final depende de qual foi
a última amostra vista.**

# Resumo dos resultados

A tabela é montada em código a partir das mesmas variáveis usadas nas análises acima, para que
não haja divergência entre o que foi calculado e o que está reportado.

| # | Quantidade | Valor |
|:-:|:---|:---|
| 1 | Exercício 1: $\mathbf{w}$ e $b$ finais | $\mathbf{w} = [0{,}050497,\ 0{,}028872]$, $b = -0{,}25$ |
| 2 | Exercício 1: épocas até a convergência | **26** (atualizações até a época 25; a 26ª é a passada limpa) |
| 3 | Exercício 1: acurácia final | **100,00%** |
| 4 | Exercício 1: épocas e acurácia final com $\eta = 1{,}0$ | **37** épocas, **100,00%** |
| 5 | Exercício 2: $\mathbf{w}$ e $b$ finais | $\mathbf{w} = [0{,}054484,\ 0{,}048043]$, $b = -0{,}07$ |
| 6 | Exercício 2: acurácia dos pesos finais | **50,15%** |
| 7 | Exercício 2: acurácia dos pesos pocket | **71,10%** ($\mathbf{w} = [0{,}010664,\ 0{,}008727]$, $b = -0{,}07$) |
| 8 | Exercício 2: época do melhor pocket | **86** |

Conferência gerada pelo código:

In [ ]:
def fmt_wb(w, b, casas=6):
    return f"w = [{w[0]:.{casas}f}, {w[1]:.{casas}f}],  b = {b:.4f}"

linhas_resumo = [
    (1, "Exercício 1 — w e b finais",
        fmt_wb(res1["w"], res1["b"])),
    (2, "Exercício 1 — épocas até a convergência",
        f"{res1['epocas']} épocas  (atualizações até a época {res1['epocas'] - 1}; "
        f"a época {res1['epocas']} é a passada limpa)"),
    (3, "Exercício 1 — acurácia final",
        f"{acuracia(X1, y1, res1['w'], res1['b'])*100:.2f}%"),
    (4, "Exercício 1 — épocas e acurácia final com eta = 1,0",
        f"{res1_eta1['epocas']} épocas,  "
        f"{acuracia(X1, y1, res1_eta1['w'], res1_eta1['b'])*100:.2f}%"),
    (5, "Exercício 2 — w e b finais",
        fmt_wb(res2["w"], res2["b"])),
    (6, "Exercício 2 — acurácia dos pesos finais",
        f"{acc_final2*100:.2f}%"),
    (7, "Exercício 2 — acurácia dos pesos pocket",
        f"{pk['acc']*100:.2f}%   ({fmt_wb(pk['w'], pk['b'])})"),
    (8, "Exercício 2 — época em que ocorreu o melhor pocket",
        f"época {pk['epoca']}"),
]

resumo = pd.DataFrame(linhas_resumo, columns=["#", "Quantidade", "Valor"]).set_index("#")
with pd.option_context("display.max_colwidth", 200):
    display(resumo.style.set_properties(**{"text-align": "left"}))

### Leitura dos números

**Exercício 1.** Com as nuvens a ~7 desvios-padrão uma da outra, o perceptron encontra uma
reta separadora em 73 atualizações, ao longo de 26 épocas. É o caso coberto pelo teorema de
convergência: o número total de erros é limitado. O $\eta$ não muda o que o algoritmo *pode*
fazer, só o peso relativo da inicialização. Com $\eta = 1{,}0$ a inicialização é apagada e o
treino coincide com a partida em zero (37 épocas). A partir do zero, $\eta$ é um fator comum que
não afeta nada.

**Exercício 2.** Com as nuvens a ~1,15 desvio-padrão, não há reta sem erros, e o laço nunca
para. Os pesos finais acertam 50,15%: a fronteira fica presa perto da origem, porque $b$ anda 5
vezes mais devagar que $\mathbf{w}$, e a cada época a nuvem inteira é passada de um lado para o
outro. O pocket guarda o instante em que a reta atravessa a nuvem e chega a 71,10%, a 0,15 ponto
percentual da melhor reta possível nesta amostra (71,25%; teórico 71,81%).

**Conclusão.** O perceptron não é "ruim" no Exercício 2: ele encontra, de passagem, uma reta
praticamente ótima. O defeito está no **critério de parada e na escolha do iterado**, que
pressupõem separabilidade. Quando ela falta, nem mais épocas nem um $\eta$ menor ajudam. É
preciso mudar o que se otimiza, e é essa a motivação para as perdas contínuas e, adiante, para
redes com mais de uma camada.